<a href="https://colab.research.google.com/github/piaseckazaneta/Python/blob/main/Co_naprawd%C4%99_decyduje_o_odej%C5%9Bciu_z_firmy.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [14]:
import pandas as pd
import numpy as np
import duckdb

In [10]:
ogolne = pd.read_csv("0.df_ogolne.csv")
zespol = pd.read_csv("1.df_zespol.csv")
tech = pd.read_csv("2.df_technologie.csv")
projekty = pd.read_csv("3.df_projekty.csv")
org_pracy = pd.read_csv("4.df_organizacja_pracy.csv")
well_being = pd.read_csv("5.df_well_being.csv")
przyszlosc = pd.read_csv("6.df_przyszlosc.csv")

tables = [ogolne, zespol, tech, projekty, org_pracy, well_being, przyszlosc]

In [13]:
for table in tables:
  print(table.shape)
  print(table.columns)
  print(table.head(2))


(297, 22)
Index(['id', 'stanowisko', 'płeć', 'miejsce zamieszkania',
       'kierunek wykształcenia', 'lata doświadczenia',
       'poziom stanowiska (seniority)', 'ile lat pracujesz na tym stanowisku',
       'wymiar pracy', 'wielkość firmy, w której pracujesz',
       'w jakim modelu realizujesz swoje projekty',
       'jaki kapitał ma firma, w której pracujesz',
       'czy przed objęciem obecnego stanowiska pracowałeś/aś już w branży it lub analitycznej',
       'wykształcenie__studiowała/em, ale nie ukończyła/em',
       'wykształcenie__doktorat', 'wykształcenie__inżynier',
       'wykształcenie__licencjat', 'wykształcenie__magister',
       'wykształcenie__studia podyplomowe', 'wykształcenie__zawodowe',
       'wykształcenie__średnie', 'rodzic'],
      dtype='object')
   id    stanowisko       płeć                        miejsce zamieszkania  \
0   0  data analyst  mężczyzna  duże miasto (powyżej 500 tys. mieszkańców)   
1   1  data analyst    kobieta  duże miasto (powyżej 500 ty

In [15]:
# Inicjalizacja DuckDB w pamięci
con = duckdb.connect()

In [18]:
# Rejestracja DataFrame'ów w silniku DuckDB
con.register('ogolne', ogolne)
con.register('zespol', zespol)
con.register('tech', tech)
con.register('projekty', projekty)
con.register('org_pracy', org_pracy)
con.register('well_being', well_being)
con.register('przyszlosc', przyszlosc)

In [20]:
query = """
CREATE OR REPLACE TABLE master_data AS
SELECT *
FROM ogolne
LEFT JOIN projekty USING (id)
LEFT JOIN well_being USING (id)
LEFT JOIN zespol USING (id)
LEFT JOIN przyszlosc USING (id)
LEFT JOIN tech USING (id)
LEFT JOIN org_pracy USING (id);
"""

con.execute(query)
df_master = con.execute("SELECT * FROM master_data").df()
print(f"Kształt: {df_master.shape}")

Kształt: (297, 169)


In [21]:
df_master.head()

,id,stanowisko,płeć,miejsce zamieszkania,kierunek wykształcenia,lata doświadczenia,poziom stanowiska (seniority),ile lat pracujesz na tym stanowisku,wymiar pracy,"wielkość firmy, w której pracujesz",...,benefity | dofinansowanie posiłków / karta lunchowa,benefity | dofinansowanie zdrowia fizycznego lub psychicznego,benefity | dostęp do platform edukacyjnych,benefity | mentoring / programy rozwojowe,benefity | możliwość pracy z dowolnego miejsca,benefity | dodatkowe dni wolne,benefity | opcje na udziały / akcje w firmie,benefity | spotkania zespołowe / wyjazdy integracyjne,benefity | budżet na home office / sprzęt do pracy zdalnej,benefity | dni wolne od spotkań
0,0,data analyst,mężczyzna,duże miasto (powyżej 500 tys. mieszkańców),"Inne techniczne (np. inżynieria, automatyka, f...",11-15 lat,mid / regular,do 1 roku,pełny etat (ok. 40 godzin tygodniowo),duża firma (powyżej 500 pracowników),...,0,0,1,1,1,0,0,1,0,0
1,1,data analyst,kobieta,duże miasto (powyżej 500 tys. mieszkańców),"Inne techniczne (np. inżynieria, automatyka, f...",6–10 lat,mid / regular,do 1 roku,pełny etat (ok. 40 godzin tygodniowo),duża firma (powyżej 500 pracowników),...,0,0,0,0,0,0,0,0,0,0
2,2,data engineer,kobieta,duże miasto (powyżej 500 tys. mieszkańców),Ekonomiczne / finansowe,6–10 lat,junior,1-2 lata,pełny etat (ok. 40 godzin tygodniowo),duża firma (powyżej 500 pracowników),...,0,1,0,0,0,1,0,1,0,0
3,3,data analyst,mężczyzna,małe miasto (poniżej 100 tys. mieszkańców),Informatyczne,11-15 lat,manager / team leader,3-4 lata,pełny etat (ok. 40 godzin tygodniowo),duża firma (powyżej 500 pracowników),...,1,0,0,1,0,1,0,1,0,0
4,4,business analyst,kobieta,duże miasto (powyżej 500 tys. mieszkańców),Ekonomiczne / finansowe,3–5 lat,mid / regular,3-4 lata,pełny etat (ok. 40 godzin tygodniowo),duża firma (powyżej 500 pracowników),...,0,0,0,0,1,0,0,0,0,0


In [22]:
# 1. Zmienna kluczowa: Ile osób rozważa odejście z pracy?
col_odejscie = 'czy obecnie rozważasz zmianę pracy'
print("--- ROZKŁAD DECYZJI O ZMIANIE PRACY ---")
print(df_master[col_odejscie].value_counts(dropna=False))
print("\nProcentowo:")
print(df_master[col_odejscie].value_counts(normalize=True).round(3) * 100)

# 2. Zarobki: Podstawowe statystyki opisowe
col_zarobki = 'zarobki'
print("\n--- STATYSTYKI ZAROBKÓW ---")
print(df_master[col_zarobki].describe().round(2))

# 3. Braki danych w kluczowych metrykach
print("\n--- BRAKI DANYCH W WYBRANYCH KOLUMNACH ---")
print(df_master[[col_odejscie, col_zarobki, 'w jaki stopniu jesteś zadowolona/y ze swojej pracy']].isna().sum())

--- ROZKŁAD DECYZJI O ZMIANIE PRACY ---
czy obecnie rozważasz zmianę pracy
tak – rozglądam się, ale bez presji    124
nie                                    123
tak – aktywnie szukam nowej pracy       25
trudno powiedzieć                       22
None                                     3
Name: count, dtype: int64

Procentowo:
czy obecnie rozważasz zmianę pracy
tak – rozglądam się, ale bez presji    42.2
nie                                    41.8
tak – aktywnie szukam nowej pracy       8.5
trudno powiedzieć                       7.5
Name: proportion, dtype: float64

--- STATYSTYKI ZAROBKÓW ---
count      267.00
mean      9455.66
std       4033.32
min       3834.00
25%       6700.00
50%       8500.00
75%      11000.00
max      31000.00
Name: zarobki, dtype: float64

--- BRAKI DANYCH W WYBRANYCH KOLUMNACH ---
czy obecnie rozważasz zmianę pracy                     3
zarobki                                               30
w jaki stopniu jesteś zadowolona/y ze swojej pracy     3
dtype: in